In [ ]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.external_data_encyclopedia import ExternalDataEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable
from boulder_statistics.refinement_plus.inspect_tools import InspectTools

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

ed = ExternalDataEncyclopedia(
    external_data_path=Path(r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data")
)

In [ ]:
exports_folder = Path("refine_plus_export_pool")
plot_export_folder = Path(".plots")

plot_export_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
from pathlib import Path

from polars.lazyframe.frame import LazyFrame

def inspect_data_segment(segment_name : str, column_name : str, collect_all = False) -> None:
    net_export_path: Path = plot_export_folder / f"{segment_name} {column_name} net.png"
    pan_export_path: Path = plot_export_folder / f"{segment_name} {column_name} pan.png"
    if net_export_path.exists() and pan_export_path.exists():
        print(f"Skipping data segment {segment_name} for column {column_name} as both plots already exist")
        return

    print(f"Exporting data segment {segment_name} for column {column_name}")
    segment: LazyFrame = pl.scan_parquet(exports_folder / segment_name).select("i", "j", "face", column_name)
    
    populated_segment: LazyFrame = segment.filter(
        pl.col(column_name).is_not_nan(), pl.col(column_name).is_not_null(), pl.col(column_name) != -9999.0
    )

    sample : pl.Series = populated_segment.collect()[column_name] if collect_all else populated_segment.filter(
        pl.col("i") < 10).collect()[column_name]
    
    faces: Dict[str, np.ndarray] = InspectTools.extract_column_as_faces(segment, column_name)

    # === Cubemap Net ===
    net = InspectTools.faces_to_cubemap_net(faces)

    plt.figure(figsize=(16, 8))
    plt.imshow(net, vmin=sample.min(), vmax=sample.max())
    plt.colorbar()
    plt.savefig(net_export_path)

    # === PAN ===
    pan = InspectTools.cubemap_to_equirectangular(faces, output_resolution=(2048, 1024))

    plt.figure(figsize=(16, 8))
    plt.imshow(pan, vmin=sample.min(), vmax=sample.max())
    plt.colorbar()
    plt.savefig(pan_export_path)

In [ ]:
from typing import List, Tuple

from boulder_statistics.refinement_plus.bulk_parse_data_tir_maps import TIR_MEASUREMENT_NAMES
from boulder_statistics.refinement_plus.bulk_parse_data_vnir_maps import VNIR_MEASUREMENT_NAMES

columns_to_plot : List[Tuple[str, str, bool]] = [
    # == TIR ===
    *[("01_add_tir_maps_detailed_survey", f"detailed_survey {c}", False)
        for c in TIR_MEASUREMENT_NAMES],

    *[("01_add_tir_maps_recona", f"recona {c}", True)
        for c in TIR_MEASUREMENT_NAMES],

    *[("01_add_tir_maps_reconb", f"reconb {c}", True)
        for c in TIR_MEASUREMENT_NAMES],

    # == VNIR ===
    *[("02_add_vnir_maps_detailed_survey", f"detailed_survey {c}", False)
        for c in VNIR_MEASUREMENT_NAMES],

    *[("02_add_vnir_maps_recona", f"recona {c}", True)
        for c in VNIR_MEASUREMENT_NAMES],

    *[("02_add_vnir_maps_reconc", f"reconc {c}", True)
        for c in VNIR_MEASUREMENT_NAMES],
]

for sn, cn, ca in columns_to_plot:
    try:
        inspect_data_segment(sn, cn, ca)
    except Exception as e:
        print(e)